<a href="https://colab.research.google.com/github/AbdouSalamSisawo/TensorOrbit-EDA-Bootcamp/blob/main/day-2-numpy-matplotlib/Day2_NumPy_Matplotlib_v2%20(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 2 — NumPy + Matplotlib
### TensorOrbit EDA Bootcamp | 22 August 2026

This notebook is built for **live-coding** — type it out with the class rather than just running it. Markdown cells are your talking points; code cells are what you type/run live.

**Note for this session:** the group includes both Computer Science and non-CS students. Content is trimmed to core concepts everyone can follow, and the second half (Matplotlib) gets more time — visualization is the main focus today.

**Dataset:** `sustainability_cities.csv`, loaded directly from GitHub so it works every time, no upload needed.

## Recap of Day 1

Remind the class:
- Yesterday we used **Pandas** to load, clean, and explore a sustainability dataset.
- Today: **NumPy** — the engine underneath Pandas — briefly, then **Matplotlib** to turn numbers into pictures. That second half is where we'll spend most of our time today.
- By the end of today, you'll take a question -> find the answer -> show it as a chart.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Loaded directly from GitHub so it works in any Colab session, no upload needed
df = pd.read_csv('https://raw.githubusercontent.com/AbdouSalamSisawo/TensorOrbit-EDA-Bootcamp/main/day-2-numpy-matplotlib/sustainability_cities.csv')
df.head()

---
# Part 1: NumPy

## Why NumPy?
Ask: *"If I wanted to double every number in a list of a million values, how would you do it in plain Python?"* -> a for-loop.

NumPy does this **without an explicit loop** (vectorisation) — faster, and shorter to write. Pandas columns are actually NumPy arrays underneath — this is *why* Pandas could do fast math on entire columns yesterday.

## Creating arrays

In [ ]:
a = np.array([1, 2, 3, 4, 5])
print(a)

zeros = np.zeros(5)
range_arr = np.arange(0, 10, 2)

print(zeros)
print(range_arr)

**Talking point:** connect straight to the dataset — pull a real column out as a NumPy array.

In [ ]:
co2 = df['co2_emissions_tons_per_capita'].to_numpy()
co2[:10]

## Indexing & slicing (1D)

We're keeping this to 1D today — one column's worth of values at a time. That's all we need for the stats and charts ahead.

In [ ]:
print(co2[0])       # first value
print(co2[:5])      # first 5
print(co2[-3:])     # last 3

## Vectorised math (no loops!)

In [ ]:
recycle = df['recycling_rate_pct'].to_numpy()
renewable = df['renewable_energy_pct'].to_numpy()

combined_score = (recycle + renewable) / 2
print(combined_score[:10])

## Boolean / fancy indexing, np.where()

**Talking point:** this is the NumPy engine behind Pandas boolean filtering they learned yesterday (`df[df['x'] > 5]`).

In [ ]:
high_co2_mask = co2 > 3
print(high_co2_mask[:10])
print(co2[high_co2_mask][:10])   # only the high-CO2 values

# np.where: label values instead of just filtering
labels = np.where(co2 > 3, 'high', 'low')
print(labels[:10])

## Sorting arrays

In [ ]:
print(np.sort(co2)[:5])   # 5 lowest
print(np.sort(co2)[-5:])  # 5 highest

## Statistical functions

In [ ]:
print('Mean CO2:', np.nanmean(co2))
print('Median CO2:', np.nanmedian(co2))
print('Std Dev CO2:', np.nanstd(co2))

We use `np.nanmean()` etc. instead of plain `np.mean()` because this column has missing values — a plain `np.mean()` would return `NaN` for the whole result. Links back to yesterday's `.isnull().sum()`.

## Percentiles & quartiles

In [ ]:
q1 = np.nanpercentile(co2, 25)
q3 = np.nanpercentile(co2, 75)
print('Q1:', q1, '| Q3:', q3)

## Outlier detection — concept only

**Talking point (no code to type here — just explain):**

One extreme value can badly distort an average or a trendline. Two common ways to define an outlier mathematically:

- **Z-score:** `z = (x - mean) / std` — a point is often flagged as an outlier when `|z| > 3`.
- **IQR:** `IQR = Q3 - Q1` — a point is often flagged as an outlier when it falls below `Q1 - 1.5*IQR` or above `Q3 + 1.5*IQR`.

We already have Q1 and Q3 above, so the IQR formula is just subtraction — no need to hand-code the filtering logic today. In a minute, we'll **see** these outliers directly with a boxplot instead — visualizing them is faster and clearer for a mixed audience than deriving the formula in code.

## Correlation: np.corrcoef()

In [ ]:
clean = df.dropna(subset=['co2_emissions_tons_per_capita', 'recycling_rate_pct'])
corr_matrix = np.corrcoef(clean['co2_emissions_tons_per_capita'], clean['recycling_rate_pct'])
print('Correlation coefficient:', corr_matrix[0, 1])

**Reading it:** close to **+1** = strong positive relationship, close to **-1** = strong negative, close to **0** = weak/no linear relationship.

**Correlation vs. causation (concept only):** ask the class — *if cities with more renewable energy also have less CO2, does renewable energy cause lower CO2, or could both be caused by something else (like wealth, policy, or city size)?* Correlation tells us variables move together; it never tells us *why* on its own.

---
# Part 2: Matplotlib — today's main focus

## Why visualise?
We just calculated a few numbers — but a table doesn't *land* the way a picture does. This is also the exact skill Day 4's presentations will need. We'll spend most of the rest of today here.

## Line chart — trend over time

In [ ]:
yearly_avg = df.groupby('year')['co2_emissions_tons_per_capita'].mean()

plt.figure(figsize=(7,4))
plt.plot(yearly_avg.index, yearly_avg.values, marker='o', color='seagreen')
plt.title('Average CO2 Emissions per Capita by Year')
plt.xlabel('Year')
plt.ylabel('CO2 (tons per capita)')
plt.grid(True)
plt.show()

## Line chart — comparing two trends on one chart

**New today:** plotting two series together with a legend is one of the most useful visualization skills — comparing two stories on one chart.

In [ ]:
yearly_renew = df.groupby('year')['renewable_energy_pct'].mean()
yearly_recycle = df.groupby('year')['recycling_rate_pct'].mean()

plt.figure(figsize=(7,4))
plt.plot(yearly_renew.index, yearly_renew.values, marker='o', label='Renewable Energy %', color='seagreen')
plt.plot(yearly_recycle.index, yearly_recycle.values, marker='s', label='Recycling Rate %', color='darkorange')
plt.title('Renewable Energy % vs Recycling Rate % Over Time')
plt.xlabel('Year')
plt.ylabel('Percent')
plt.legend()
plt.grid(True)
plt.show()

## Bar chart — comparing categories

In [ ]:
top10 = df.groupby('city')['renewable_energy_pct'].mean().sort_values(ascending=False).head(10)

plt.figure(figsize=(8,4))
plt.bar(top10.index, top10.values, color='steelblue')
plt.title('Top 10 Cities by Renewable Energy %')
plt.xlabel('City')
plt.ylabel('Renewable Energy (%)')
plt.xticks(rotation=45, ha='right')
plt.show()

## Histogram — distribution of a single variable

In [ ]:
plt.figure(figsize=(7,4))
plt.hist(df['co2_emissions_tons_per_capita'].dropna(), bins=15, color='darkorange', edgecolor='black')
plt.title('Distribution of CO2 Emissions per Capita')
plt.xlabel('CO2 (tons per capita)')
plt.ylabel('Frequency')
plt.show()

## Scatter plot — relationship between two variables

In [ ]:
plt.figure(figsize=(7,4))
plt.scatter(df['renewable_energy_pct'], df['co2_emissions_tons_per_capita'], alpha=0.6, color='purple')
plt.title('Renewable Energy % vs CO2 Emissions per Capita')
plt.xlabel('Renewable Energy (%)')
plt.ylabel('CO2 (tons per capita)')
plt.show()

## Boxplot — seeing outliers directly

**This is where the Z-score/IQR concept from earlier becomes visible.** No formula needed — the dots beyond the whiskers ARE the outliers.

In [ ]:
plt.figure(figsize=(5,5))
plt.boxplot(df['co2_emissions_tons_per_capita'].dropna(), vert=True)
plt.title('CO2 Emissions per Capita — Boxplot')
plt.ylabel('CO2 (tons per capita)')
plt.show()

## Pie chart — share of a whole

**Talking point:** pie charts work best with few categories (3-6) that sum to a meaningful whole — use sparingly.

In [ ]:
country_counts = df['country'].value_counts().head(5)

plt.figure(figsize=(6,6))
plt.pie(country_counts.values, labels=country_counts.index, autopct='%1.0f%%')
plt.title('Sample Distribution: Top 5 Countries by Record Count')
plt.show()

## Making charts easier to read: color and annotation

**New today — worth the extra few minutes since visualization is our focus.** A couple of small touches make a chart much clearer to an audience.

In [ ]:
plt.figure(figsize=(7,4))
bars = plt.bar(top10.index[:5], top10.values[:5], color=['#2D6A4F' if v > 50 else '#95D5B2' for v in top10.values[:5]])
plt.title('Top 5 Cities by Renewable Energy % (highlighting >50%)')
plt.xlabel('City')
plt.ylabel('Renewable Energy (%)')
plt.xticks(rotation=30, ha='right')

# Annotate each bar with its value
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.5, f'{height:.0f}%', ha='center', fontsize=9)

plt.show()

**Talking point:** color can carry meaning (here, green = above 50%), and a data label removes the need for the audience to squint at the axis. Small additions, big clarity gain.

## Subplots — multiple charts side by side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(yearly_avg.index, yearly_avg.values, marker='o', color='seagreen')
axes[0].set_title('Avg CO2 by Year')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('CO2 (tons per capita)')

axes[1].scatter(df['renewable_energy_pct'], df['co2_emissions_tons_per_capita'], alpha=0.6, color='purple')
axes[1].set_title('Renewable % vs CO2')
axes[1].set_xlabel('Renewable Energy (%)')
axes[1].set_ylabel('CO2 (tons per capita)')

plt.tight_layout()
plt.show()

---
## Guided Practice

Take **2-3 findings** and turn each into a clearly labelled chart, choosing the chart type that fits the question. Every chart needs: a title, axis labels, and (if more than one series) a legend.

Prompts to put on the board:
1. Which city had the highest average water usage per capita across all years? Show it.
2. Is there a relationship between green space % and recycling rate %? Show it.
3. How has the average recycling rate changed year over year? Show it.

Work in the empty cells below.

In [ ]:
# Practice 1


In [ ]:
# Practice 2


In [ ]:
# Practice 3


---
## Wrap-up
- Preview Day 3: tomorrow you'll combine Pandas + NumPy + Matplotlib into one full workflow on a *business* dataset, then get your capstone brief.